# run.1 — Limitation Re-Test: Tiny-ImageNet-C (200 classes)

## Purpose
run found BMIA-A=12.8%, BMIA-F=22.4% on Tiny-ImageNet-C (200 classes).  
Root cause: τ=0.10 rarely triggers at 200 classes (uniform = 0.5%/class) → λ relaxes too fast.  
This notebook tests whether τ=10/K=0.05 and run-updated variants fix the limitation.

## Dataset format (ImageFolder, NOT .npy)
- `luckyhathaway/tiny-imagenet-c` → `corruption/severity/classname/*.jpg`  OR  `corruption/classname/*.jpg`
- `akash2sharma/tiny-imagenet`     → `tiny-imagenet-200/train/classname/*.jpg`

## Methods tested (8)
| Method | Config | Baseline |
|--------|--------|----------|
| TENT   | entropy min | — |
| SAR    | filtered entropy | — |
| EATA   | Fisher-anchored entropy | — |
| BMIA-F | λ=1.0, τ=0.10 | run: 22.4% |
| BMIA-A | λ_init=0.5, τ=0.10 | run: 12.8% |
| BMIA-A-V3 | λ_init=0.1, accum=2, τ=0.10 | run update |
| BMIA-A-Scaled | λ_init=0.5, τ=0.05 | proposed fix |
| BMIA-F-Scaled | λ=1.0, τ=0.05 | proposed fix |

## Kaggle setup
1. Add Data → `luckyhathaway/tiny-imagenet-c`
2. Add Data → `akash2sharma/tiny-imagenet`
3. Accelerator: **GPU T4 x2**, Internet: **ON**

## Estimated time
- Training 30 epochs (DataParallel T4×2): ~12–15 min  
- 80 TTA runs: ~15–20 min  
- **Total: ~30 min**

In [ ]:
# ================================================================
# Cell 0 — Dataset Verification
# Must pass 100% before any training starts
# ================================================================
import os

print('=' * 60)
print('DATASET VERIFICATION')
print('=' * 60)

# ── 1. Find Tiny-ImageNet-C ──────────────────────────────────────
DATA_ROOT = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'gaussian_noise' in dirs and 'defocus_blur' in dirs:
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    raise RuntimeError(
        'Tiny-ImageNet-C NOT FOUND.\n'
        'Add dataset via Add Data: luckyhathaway/tiny-imagenet-c'
    )
print(f'[OK] Tiny-ImageNet-C root : {DATA_ROOT}')

# Detect severity folder format
gn_path = os.path.join(DATA_ROOT, 'gaussian_noise')
gn_subs = sorted(os.listdir(gn_path))
print(f'     gaussian_noise/       : {gn_subs[:6]}...')

SEVERITY = 5
if str(SEVERITY) in gn_subs:
    CORR_PATH_TEMPLATE = os.path.join(DATA_ROOT, '{corruption}', str(SEVERITY))
    print(f'     Format detected       : corruption/{SEVERITY}/classname/')
else:
    CORR_PATH_TEMPLATE = os.path.join(DATA_ROOT, '{corruption}')
    print(f'     Format detected       : corruption/classname/')

# ── 2. Find Tiny-ImageNet clean train ───────────────────────────
TRAIN_ROOT = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if not root.endswith('/train'):
        continue
    if DATA_ROOT in root:          # skip corruption dataset subtrees
        continue
    cls_dirs = [d for d in os.listdir(root)
                if os.path.isdir(os.path.join(root, d))]
    if len(cls_dirs) >= 190:       # expect 200 WordNet-ID folders
        TRAIN_ROOT = root
        print(f'[OK] Tiny-ImageNet train  : {TRAIN_ROOT}')
        print(f'     Class folders         : {len(cls_dirs)}')
        break

if TRAIN_ROOT is None:
    raise RuntimeError(
        'Tiny-ImageNet train NOT FOUND.\n'
        'Add dataset via Add Data: akash2sharma/tiny-imagenet'
    )

# ── 3. Verify all 5 corruption folders ──────────────────────────
CORRUPTIONS = ['gaussian_noise', 'defocus_blur', 'snow',
               'contrast', 'jpeg_compression']
print()
for c in CORRUPTIONS:
    p  = CORR_PATH_TEMPLATE.format(corruption=c)
    ok = os.path.isdir(p)
    print(f'  [{"OK" if ok else "MISSING"}] {c:<22} → {p}')
    if not ok:
        raise RuntimeError(f'Corruption folder missing: {p}')

print()
print('[PASS] All datasets verified — ready to run.')

In [ ]:
# ================================================================
# Cell 1 — Imports & Constants
# ================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time, json, copy, gc
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.benchmark = True   # faster on fixed 64×64 input

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Device : {device}  |  GPUs available : {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# ── Experiment constants ────────────────────────────────────────
NUM_CLASSES = 200
TTA_BATCH   = 64
ACCUM       = 2          # V3 effective batch = 128
EVAL_BATCH  = 256
N_SAMPLES   = 5000
LR_LIST     = [0.005, 0.05]
TAU_STD     = 0.10
TAU_SCALED  = 10.0 / NUM_CLASSES   # = 0.05 for 200 classes

# Tiny-ImageNet normalisation — same as run
TINY_MEAN = (0.4802, 0.4481, 0.3975)
TINY_STD  = (0.2302, 0.2265, 0.2262)

METHODS = [
    'tent',
    'sar',
    'eata',
    'bmia_fixed',            # λ=1.0,  τ=0.10          (run baseline 22.4%)
    'bmia_adaptive',         # λ=0.5,  τ=0.10          (run baseline 12.8%)
    'bmia_adaptive_v3',      # λ=0.1,  τ=0.10, accum=2 (run update)
    'bmia_adaptive_scaled',  # λ=0.5,  τ=0.05          (proposed fix)
    'bmia_fixed_scaled',     # λ=1.0,  τ=0.05          (proposed fix)
]

print(f'\nNUM_CLASSES : {NUM_CLASSES}')
print(f'TAU_STD     : {TAU_STD}   TAU_SCALED : {TAU_SCALED} (=10/{NUM_CLASSES})')
print(f'LR_LIST     : {LR_LIST}')
print(f'Total runs  : {len(METHODS)} × {len(LR_LIST)} × {len(CORRUPTIONS)} = '
      f'{len(METHODS)*len(LR_LIST)*len(CORRUPTIONS)}')

In [ ]:
# ================================================================
# Cell 2 — Train ResNet-18 on Tiny-ImageNet
#
# Architecture (same as run):
#   3×3 conv1 (stride=1) + Identity maxpool for 64×64 images
#   Default ResNet-18 uses 7×7 conv1 (stride=2) — wrong for 64×64
#
# Speed:
#   DataParallel on T4×2 for training only (~2× speedup)
#   TTA always uses single-GPU model to avoid state_dict key conflict
# ================================================================
train_transform = transforms.Compose([
    transforms.Resize(64),
    transforms.RandomCrop(64, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(TINY_MEAN, TINY_STD),
])

trainset    = torchvision.datasets.ImageFolder(TRAIN_ROOT, transform=train_transform)
trainloader = DataLoader(trainset, batch_size=128, shuffle=True,
                         num_workers=2, pin_memory=True)
print(f'Train samples : {len(trainset)}  |  Classes : {len(trainset.classes)}')
assert len(trainset.classes) == NUM_CLASSES, \
    f'Expected {NUM_CLASSES} classes, got {len(trainset.classes)}'

CLASS_TO_IDX = trainset.class_to_idx   # WordNetID → int, used for test remapping

# ── Model (single GPU, used for TTA throughout) ─────────────────
model = torchvision.models.resnet18(weights=None)
model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc      = nn.Linear(512, NUM_CLASSES)
model         = model.to(device)

# ── DataParallel wrapper for training only ──────────────────────
# model_dp shares parameters with model (same tensors in-place)
# optimizer uses model.parameters() — gradients aggregate on GPU 0
# After training: del model_dp, save SOURCE_STATE from model (no 'module.' prefix)
if N_GPUS > 1:
    model_dp = nn.DataParallel(model)
    print(f'DataParallel: {N_GPUS} GPUs for training')
else:
    model_dp = model
    print('Single-GPU training')

criterion       = nn.CrossEntropyLoss()
optimizer_train = optim.SGD(model.parameters(), lr=0.1,
                            momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_train, T_max=30)

print('\nTraining 30 epochs...')
t0 = time.time()
for epoch in range(30):
    model_dp.train()
    correct, total = 0, 0
    for X_b, y_b in trainloader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_train.zero_grad()
        out  = model_dp(X_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer_train.step()
        correct += (out.argmax(1) == y_b).sum().item()
        total   += y_b.size(0)
    scheduler.step()
    elapsed = (time.time() - t0) / 60
    eta     = elapsed / (epoch + 1) * (30 - epoch - 1)
    print(f'  Epoch {epoch+1:2d}/30  train {100*correct/total:.1f}%  elapsed {elapsed:.1f}min  ETA {eta:.1f}min')

# ── Save source state from UNWRAPPED model ───────────────────────
model.eval()
SOURCE_STATE = copy.deepcopy(model.state_dict())

# ── Free training resources before TTA ──────────────────────────
del model_dp, trainloader, trainset   # release DataParallel + worker processes
gc.collect()
torch.cuda.empty_cache()

print(f'\nTraining done in {(time.time()-t0)/60:.1f} min')
print('SOURCE_STATE saved. Training resources freed.')

In [ ]:
# ================================================================
# Cell 3 — Data Loader + Source Accuracy Check
#
# Notes:
#  - Uses ImageFolder (NOT .npy) — matches luckyhathaway/tiny-imagenet-c format
#  - Explicit label remap: test class_to_idx → train CLASS_TO_IDX
#    (both use alphabetical WordNet-ID ordering, so remap is identity in practice;
#     kept explicit for safety against any ordering difference)
#  - n_samples=5000 from 10,000 total → first 100 classes alphabetically
#    (consistent with run baseline for fair comparison)
# ================================================================
test_transform = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize(TINY_MEAN, TINY_STD),
])

def load_corruption(corruption, n_samples=N_SAMPLES):
    corr_dir = CORR_PATH_TEMPLATE.format(corruption=corruption)
    dataset  = torchvision.datasets.ImageFolder(corr_dir, transform=test_transform)

    # Build test_idx → train_idx remap
    test_to_train = {
        test_idx: CLASS_TO_IDX[cls_name]
        for cls_name, test_idx in dataset.class_to_idx.items()
        if cls_name in CLASS_TO_IDX
    }
    # Classes present in test but missing in train
    missing_cls = set(dataset.class_to_idx.keys()) - set(CLASS_TO_IDX.keys())
    if missing_cls:
        print(f'  WARNING [{corruption}]: {len(missing_cls)} classes in test '
              f'not found in train → labels kept as-is')

    n_load  = min(n_samples, len(dataset))
    loader  = DataLoader(
        torch.utils.data.Subset(dataset, list(range(n_load))),
        batch_size=EVAL_BATCH, shuffle=False,
        num_workers=2, pin_memory=True
    )

    all_x, all_y = [], []
    for x, y in loader:
        y_remapped = torch.tensor(
            [test_to_train.get(yi.item(), yi.item()) for yi in y],
            dtype=torch.long
        )
        all_x.append(x)
        all_y.append(y_remapped)

    X = torch.cat(all_x)[:n_samples].to(device)
    Y = torch.cat(all_y)[:n_samples].to(device)

    n_cls_covered = len(torch.unique(Y).tolist())
    print(f'  Loaded {X.size(0):5d} samples  |  {n_cls_covered:3d}/{NUM_CLASSES} classes covered')
    return X, Y

# ── Source accuracy check ────────────────────────────────────────
print('Source accuracy check on gaussian_noise (500 samples):')
model.eval()
X_chk, y_chk = load_corruption('gaussian_noise', n_samples=500)
with torch.no_grad():
    src_acc = (model(X_chk).argmax(1) == y_chk).float().mean().item()
print(f'  Source accuracy : {src_acc:.4f}  ({src_acc*100:.1f}%)')
assert src_acc > 0.02, (
    f'Source accuracy {src_acc:.4f} is too low — '
    f'possible label mismatch or failed training'
)
clean_acc = float(src_acc)   # stored for JSON
del X_chk, y_chk
gc.collect()
torch.cuda.empty_cache()
print('Data loader ready.')

In [ ]:
# ================================================================
# Cell 4 — Helpers (identical to run)
# ================================================================
def freeze_except_bn(mdl):
    """Freeze all params except BN weight and bias."""
    bn_ids = set()
    for m in mdl.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            for p in m.parameters():
                bn_ids.add(id(p))
    for p in mdl.parameters():
        p.requires_grad_(id(p) in bn_ids)


def get_metrics(mdl, X, y):
    """Evaluate accuracy, collapse stats, and MI metrics."""
    mdl.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for i in range(0, X.size(0), EVAL_BATCH):
            logits = mdl(X[i:i + EVAL_BATCH])
            probs  = torch.softmax(logits, 1)
            all_preds.append(logits.argmax(1))
            all_probs.append(probs)
    preds  = torch.cat(all_preds)
    probs  = torch.cat(all_probs)
    acc    = (preds == y[:len(preds)]).float().mean().item()
    counts = torch.bincount(preds, minlength=NUM_CLASSES).float()
    dom    = counts.max().item() / counts.sum().item()
    n_cls  = (counts > 0).sum().item()
    h_yx   = -(probs * torch.log(probs + 1e-8)).sum(1).mean().item()
    mg     = probs.mean(0)
    h_y    = -(mg * torch.log(mg + 1e-8)).sum().item()
    return {
        'acc':       round(acc, 4),
        'dom_pct':   round(dom, 4),
        'n_classes': n_cls,
        'h_y':       round(h_y, 4),
        'h_yx':      round(h_yx, 4),
        'mi':        round(h_y - h_yx, 4),
    }


def is_collapsed(r):
    """Collapse criterion (same as all previous versions)."""
    return (r['dom_pct'] > 0.15 and r['n_classes'] < 50) or r['n_classes'] < 20


print('freeze_except_bn  ✓')
print('get_metrics       ✓')
print('is_collapsed      ✓  (dom>15% & cls<50) OR cls<20')

In [ ]:
# ================================================================
# Cell 5 — TTA run_method (run structure)
#
# model is always the SINGLE-GPU unwrapped ResNet-18
# SOURCE_STATE has no 'module.' prefix → load_state_dict always works
# ================================================================
METHOD_CFG = {
    'bmia_fixed':           {'lam': 1.0, 'tau': TAU_STD,    'accum': False},
    'bmia_adaptive':        {'lam': 0.5, 'tau': TAU_STD,    'accum': False},
    'bmia_adaptive_v3':     {'lam': 0.1, 'tau': TAU_STD,    'accum': True },
    'bmia_adaptive_scaled': {'lam': 0.5, 'tau': TAU_SCALED, 'accum': False},
    'bmia_fixed_scaled':    {'lam': 1.0, 'tau': TAU_SCALED, 'accum': False},
}
FIXED_METHODS = {'bmia_fixed', 'bmia_fixed_scaled'}


def run_method(X, y, method, lr):
    # ── Reset to source ──────────────────────────────────────────
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train()
    freeze_except_bn(model)

    params      = [p for p in model.parameters() if p.requires_grad]
    opt         = optim.SGD(params, lr=lr, momentum=0.9)
    init_params = {pn: p.data.clone()
                   for pn, p in model.named_parameters() if p.requires_grad}

    cfg       = METHOD_CFG.get(method, {})
    lam       = cfg.get('lam', 0.5)
    tau       = cfg.get('tau', TAU_STD)
    use_accum = cfg.get('accum', False)
    is_fixed  = method in FIXED_METHODS

    # ── EATA: pre-compute Fisher then reset ──────────────────────
    if method == 'eata':
        fisher = {pn: torch.zeros_like(p)
                  for pn, p in model.named_parameters() if p.requires_grad}
        n_f = min(128, X.size(0))
        for j in range(0, n_f, TTA_BATCH):
            xb = X[j:j + TTA_BATCH]
            if xb.size(0) < 2:
                continue
            model.zero_grad()
            pr = torch.softmax(model(xb), 1)
            (-(pr * torch.log(pr + 1e-8)).sum(1).mean()).backward()
            for pn, p in model.named_parameters():
                if pn in fisher and p.grad is not None:
                    fisher[pn] += p.grad.data ** 2 * xb.size(0)
        for pn in fisher:
            fisher[pn] /= n_f
        # reset model so TTA starts from source
        model.load_state_dict(copy.deepcopy(SOURCE_STATE))
        model.train()
        freeze_except_bn(model)
        params = [p for p in model.parameters() if p.requires_grad]
        opt    = optim.SGD(params, lr=lr, momentum=0.9)
        # init_params already set from source above — still valid after reset

    # ── TTA loop ─────────────────────────────────────────────────
    step = TTA_BATCH * ACCUM if use_accum else TTA_BATCH

    for i in range(0, X.size(0), step):
        if use_accum:
            chunks = [
                X[i + s * TTA_BATCH: i + (s + 1) * TTA_BATCH]
                for s in range(ACCUM)
            ]
            chunks = [c for c in chunks if c.size(0) >= 4]
            if not chunks:
                continue
            xb = torch.cat(chunks)
        else:
            xb = X[i:i + TTA_BATCH]
            if xb.size(0) < 4:
                continue

        opt.zero_grad()
        probs = torch.softmax(model(xb), 1)
        ent   = -(probs * torch.log(probs + 1e-8)).sum(1)

        if method == 'tent':
            loss = ent.mean()

        elif method == 'sar':
            mask = ent < 0.4 * np.log(NUM_CLASSES)
            if mask.sum() < 2:
                continue
            loss = ent[mask].mean()

        elif method == 'eata':
            mask = ent < 0.4 * np.log(NUM_CLASSES)
            if mask.sum() < 2:
                continue
            fl = sum(
                (fisher[pn] * (p - init_params[pn]) ** 2).sum()
                for pn, p in model.named_parameters() if pn in fisher
            )
            loss = ent[mask].mean() + 2000 * fl

        else:   # all BMIA variants
            mg     = probs.mean(0)
            h_y    = -(mg * torch.log(mg + 1e-8)).sum()
            anchor = sum(
                ((p - init_params[pn]) ** 2).sum()
                for pn, p in model.named_parameters() if pn in init_params
            )
            loss = ent.mean() - h_y + lam * anchor

        loss.backward()
        opt.step()

        # Adaptive λ update (non-fixed BMIA only)
        if not is_fixed and method not in ('tent', 'sar', 'eata'):
            with torch.no_grad():
                dom = (
                    torch.bincount(probs.argmax(1), minlength=NUM_CLASSES)
                    .float().max() / xb.size(0)
                ).item()
            lam = min(10.0, lam * 1.1) if dom > tau else max(0.01, lam * 0.95)

    return get_metrics(model, X, y)


print('run_method ready.')
print(f'  Standard τ = {TAU_STD}  |  Scaled τ = {TAU_SCALED} (= 10/{NUM_CLASSES})')
print(f'  V3 accum: effective batch = {TTA_BATCH}×{ACCUM} = {TTA_BATCH*ACCUM}')

In [ ]:
# ================================================================
# Cell 6 — Run All 80 Experiments
# 8 methods × 2 lr × 5 corruptions
# ================================================================
all_results = []
t_total = time.time()

for corr in CORRUPTIONS:
    print(f'\n{"="*65}')
    X, y = load_corruption(corr)
    print(f'  Corruption: {corr}')
    print(f'  {"lr":<7} {"Method":<24} {"Acc":>6} {"Dom":>6} {"Cls":>4} {"MI":>5} {"sec":>5}')
    print(f'  {"-"*60}')

    for lr in LR_LIST:
        for method in METHODS:
            t0  = time.time()
            r   = run_method(X, y, method=method, lr=lr)
            col = is_collapsed(r)
            tag = '  *** COLLAPSED ***' if col else ''
            print(
                f'  {lr:<7} {method:<24} '
                f'{r["acc"]*100:5.1f}% '
                f'{r["dom_pct"]*100:5.1f}% '
                f'{r["n_classes"]:4d} '
                f'{r["mi"]:5.2f} '
                f'{time.time()-t0:5.0f}'
                f'{tag}'
            )
            all_results.append({
                'corruption': corr,
                'severity':   SEVERITY,
                'method':     method,
                'lr':         lr,
                **r,
                'collapsed':  col,
            })
            gc.collect()
            torch.cuda.empty_cache()

    del X, y
    gc.collect()
    torch.cuda.empty_cache()

elapsed = (time.time() - t_total) / 60
print(f'\n{"="*65}')
print(f'All done: {elapsed:.1f} min  |  {len(all_results)} results')

In [ ]:
# ================================================================
# Cell 7 — Summary vs run Baseline
# ================================================================
BASELINE = {'bmia_fixed': 22.4, 'bmia_adaptive': 12.8}


def avg_acc(m, lr):
    vals = [r['acc'] for r in all_results if r['method'] == m and r['lr'] == lr]
    return np.mean(vals) * 100 if vals else 0.0


def n_col(m, lr):
    return sum(1 for r in all_results
               if r['method'] == m and r['lr'] == lr and r['collapsed'])


print('=' * 75)
print('run.1 — Tiny-ImageNet-C Limitation Re-Test  (200 classes, Severity 5)')
print('=' * 75)

for lr in LR_LIST:
    print(f'\n─── lr = {lr} ───')
    print(f'  {"Method":<26} {"Avg Acc":>8} {"run Ref":>8} {"Change":>8} {"Collapse":>10}')
    print(f'  {"-"*65}')
    for method in METHODS:
        avg  = avg_acc(method, lr)
        nc   = n_col(method, lr)
        ref  = BASELINE.get(method)
        rs   = f'{ref:.1f}%' if ref is not None else '—'
        ds   = f'{avg - ref:+.1f}%' if ref is not None else '—'
        print(f'  {method:<26} {avg:>7.1f}% {rs:>8} {ds:>8} '
              f'{nc:>3}/{len(CORRUPTIONS)}')

print()
print('=' * 75)
print('KEY QUESTION: Does τ=0.05 fix the 200-class limitation?')
print('=' * 75)
for lr in LR_LIST:
    print(f'\n  lr = {lr}')
    print(f'    BMIA-A original  (τ=0.10) : {avg_acc("bmia_adaptive", lr):5.1f}%'
          f'  [run baseline : 12.8%]')
    print(f'    BMIA-A-V3        (τ=0.10) : {avg_acc("bmia_adaptive_v3", lr):5.1f}%'
          f'  [run update]')
    print(f'    BMIA-A-Scaled    (τ=0.05) : {avg_acc("bmia_adaptive_scaled", lr):5.1f}%'
          f'  [proposed fix]')
    print(f'    BMIA-F original  (τ=0.10) : {avg_acc("bmia_fixed", lr):5.1f}%'
          f'  [run baseline : 22.4%]')
    print(f'    BMIA-F-Scaled    (τ=0.05) : {avg_acc("bmia_fixed_scaled", lr):5.1f}%')

In [ ]:
# ================================================================
# Cell 8 — Save JSON & Print Full Output
# Paste the printed JSON back to pannel after run completes
# ================================================================
save_data = {
    'experiment':    'run.1_LIMITATION_TINYIMAGENET',
    'model':         'ResNet-18 (3x3 conv1 + Identity maxpool, seed=42, 30ep)',
    'clean_acc':     clean_acc,
    'num_classes':   NUM_CLASSES,
    'lr_list':       LR_LIST,
    'severity':      SEVERITY,
    'corruptions':   CORRUPTIONS,
    'tau_standard':  TAU_STD,
    'tau_scaled':    float(TAU_SCALED),
    'baseline':  BASELINE,
    'results':       all_results,
}

out_path = 'LIMITATION.json'
with open(out_path, 'w') as f:
    json.dump(save_data, f, indent=2, default=str)

print(f'Saved → {out_path}')
print(f'Size  → {os.path.getsize(out_path) / 1024:.1f} KB')
print()
# Print full JSON so it can be copy-pasted back to pannel
with open(out_path) as f:
    print(f.read())